In [21]:
import ctypes
import psutil

# now read the adres

# Constants
PROCESS_ALL_ACCESS = 0x1F0FFF

# Function to get the process ID
def get_process_id(process_name):
    for proc in psutil.process_iter(['pid', 'name']):
        if proc.info['name'] == process_name:
            return proc.info['pid']
    return None

# Function to read memory from a given process at a specific address
def read_memory(process_handle, address, data_type=ctypes.c_uint64):
    value = data_type()
    bytes_read = ctypes.c_size_t()
    result = ctypes.windll.kernel32.ReadProcessMemory(process_handle,ctypes.c_void_p(address),ctypes.byref(value),ctypes.sizeof(value),ctypes.byref(bytes_read))
    if result:
        return value.value
    else:
        error_code = ctypes.GetLastError()
        print(f"Failed to read memory. Error code: {error_code}")
        return None

# Main validation
process_name = "DolphinMemoryEngine.exe"
process_id = get_process_id(process_name)

# Open the process
process_handle = ctypes.windll.kernel32.OpenProcess(PROCESS_ALL_ACCESS, False, process_id)
if process_handle:
    # Base address of PsychEngine.exe
    base_address = 0x7FFB046C0000 # Qt6Core.dll
    base_offset = 0x00545380

    offsets = [0x70]

    first_pointer = base_address + base_offset
    dereferenced_address  = read_memory(process_handle, first_pointer)
    dereferenced_address_hex = (hex(dereferenced_address))
    print(dereferenced_address_hex)

    for offset in offsets:
        pointer = dereferenced_address + offset
        print(f'Pointer Addre: {hex(pointer)}')
        dereferenced_address  = read_memory(process_handle, pointer)
        print(f'Address Value: {dereferenced_address}')
        dereferenced_address_hex = (hex(dereferenced_address))
        print(dereferenced_address_hex)

0x264b5fe2b20
Pointer Addre: 0x264b5fe2b90
Address Value: 22162031247369
0x142800000009


In [16]:
val = read_memory(process_handle, 0x264B5FE2B90)

In [17]:
val

22162031247367

# Read Memory 100 times
make sure you have dolphin running, and have DME running too which is monitoring the values of score left. next up make sure to get the addresses of DME in cheat engine and get their addressses and base addres and offsets and fill them in below

In [98]:
import struct
class MemoryObject:
    def __init__(self, base_offset, offsets, value_name):
        self.base_offset = base_offset
        self.offsets = offsets
        self.value_name = value_name

# Convert 8-byte value to 4-byte value
def convert_to_4_bytes(value_8_byte):
    return value_8_byte & 0xFFFFFFFF  # Mask to keep lower 4 bytes

def to_float(val):
    return struct.unpack('f', struct.pack('I', val))[0]

# Get Process
process_name = "DolphinMemoryEngine.exe"
process_id = get_process_id(process_name)
process_handle = ctypes.windll.kernel32.OpenProcess(PROCESS_ALL_ACCESS, False, process_id)

# SETUP
BASE_ADDRESS = 0x7FFB046C0000 # keeps changing every instance
# SCORE
SCORE_LEFT = MemoryObject(0x00545380, [0x70], "Score_Left")
SCORE_RIGHT = MemoryObject(0x005454C0, [0x0, 0x290], "Score_Right")
# Ball
BALL_X_POS = None
BALL_Y_POS = None
# PLAYERS
P1_X_POS = MemoryObject(0x00545C80, [0x270, 0x148, 0x8, 0xBB0], "P1_X_Pos")
P1_Y_POS = MemoryObject(0x00545648, [0x78, 0X238, 0X898, 0XD0, 0X8A0], "P1_Y_Pos")

P2_X_POS = MemoryObject(0x00545688, [0XB8, 0X90, 0X240, 0X0], "P2_X_Pos")
P2_Y_POS = MemoryObject(0x00545750, [0x28, 0x150, 0x8, 0x38, 0x2F0], "P2_Y_Pos")

P3_X_POS = MemoryObject(0x00545690, [0X18, 0x80, 0x8, 0x330], "P3_X_Pos")
P3_Y_POS = MemoryObject(0x00545690, [0XA8, 0X458, 0X328, 0X9D0], "P3_Y_Pos")

P4_X_POS = MemoryObject(0x00545C80, [0X188, 0x50, 0x20, 0x50, 0xA0], "P4_X_Pos")
P4_Y_POS = MemoryObject(0x00545AD8, [0x148, 0x5C8, 0x328, 0x910], "P4_Y_Pos")

In [76]:
def read_value_from_memory_as_float(process_handle, base_address, mem_obj:MemoryObject, n=1):
    first_pointer = base_address + mem_obj.base_offset
    dereferenced_address_base = read_memory(process_handle, first_pointer)
    #dereferenced_address_hex = (hex(dereferenced_address_base))
    # print(dereferenced_address_hex)
    deref_addr_slot = dereferenced_address_base

    for _ in range(n):
        deref_addr_slot = dereferenced_address_base
        value_slot = 0
        for offset in mem_obj.offsets:
            pointer = deref_addr_slot + offset
            # print(f'Pointer Address: {hex(pointer)}')
            deref_addr_slot = read_memory(process_handle, pointer)
            # print(f'8-byte Address Value: {deref_addr_slot}')
            dereferenced_address_hex = (hex(deref_addr_slot))
            # print(f'8-byte Address Value (Hex): {dereferenced_address_hex}')

            # Convert to 4-byte value
            value_4_byte = convert_to_4_bytes(deref_addr_slot)
            # print(f'4-byte Address Value: {value_4_byte}')
            # print(f'4-byte Address Value (Hex): {hex(value_4_byte)}')
            value_slot = value_4_byte
        # print(f'{VALUE_NAME}: {value_slot}')
        print(f'{mem_obj.value_name}: {to_float(value_slot)}')

def read_value_from_memory_as_byte(process_handle, base_address, mem_obj:MemoryObject, n=1):
    first_pointer = base_address + mem_obj.base_offset
    dereferenced_address_base = read_memory(process_handle, first_pointer)
    #dereferenced_address_hex = (hex(dereferenced_address_base))
    # print(dereferenced_address_hex)
    deref_addr_slot = dereferenced_address_base

    for _ in range(n):
        deref_addr_slot = dereferenced_address_base
        value_slot = 0
        for offset in mem_obj.offsets:
            pointer = deref_addr_slot + offset
            # print(f'Pointer Address: {hex(pointer)}')
            deref_addr_slot = read_memory(process_handle, pointer)
            # print(f'8-byte Address Value: {deref_addr_slot}')
            dereferenced_address_hex = (hex(deref_addr_slot))
            # print(f'8-byte Address Value (Hex): {dereferenced_address_hex}')

            # Convert to 4-byte value
            value_4_byte = convert_to_4_bytes(deref_addr_slot)
            # print(f'4-byte Address Value: {value_4_byte}')
            # print(f'4-byte Address Value (Hex): {hex(value_4_byte)}')
            value_slot = value_4_byte
        # print(f'{VALUE_NAME}: {value_slot}')
        print(f'{mem_obj.value_name}: {value_slot}')

In [99]:
read_value_from_memory_as_byte(process_handle, BASE_ADDRESS, SCORE_LEFT)
read_value_from_memory_as_byte(process_handle, BASE_ADDRESS, SCORE_RIGHT)
read_value_from_memory_as_float(process_handle, BASE_ADDRESS, P1_X_POS)
read_value_from_memory_as_float(process_handle, BASE_ADDRESS, P1_Y_POS)
read_value_from_memory_as_float(process_handle, BASE_ADDRESS, P2_X_POS)
read_value_from_memory_as_float(process_handle, BASE_ADDRESS, P2_Y_POS)
read_value_from_memory_as_float(process_handle, BASE_ADDRESS, P3_X_POS)
read_value_from_memory_as_float(process_handle, BASE_ADDRESS, P3_Y_POS)
read_value_from_memory_as_float(process_handle, BASE_ADDRESS, P4_X_POS)
read_value_from_memory_as_float(process_handle, BASE_ADDRESS, P4_Y_POS)

Score_Left: 19
Score_Right: 6
P1_X_Pos: -540.2000122070312
P1_Y_Pos: 140.0
P2_X_Pos: 610.7539672851562
P2_Y_Pos: 0.0
P3_X_Pos: 385.1000061035156
P3_Y_Pos: -140.0
P4_X_Pos: -385.1000061035156
P4_Y_Pos: -140.0
